# Example 15.1 / 1: A smallest reproducer to attach

Companion notebook for **15.1 AMPLbot in ChatGPT** in the AMPL lab Documentation, section "A smallest reproducer to attach". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A smallest reproducer to attach

The model below is the kind of reproducer that prompt expects: the steel production model with
one added commitment to ship at least 8000 tons in total.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                     # steel products
param rate {PROD} > 0;        # tons per hour
param profit {PROD};          # dollars per ton
param market {PROD} >= 0;     # tons
param avail >= 0;             # hours
param min_total >= 0;         # tons

var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit, dollars

# rolling hours used fit within avail
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# total output of at least min_total tons
subject to Commitment: sum {p in PROD} Make[p] >= min_total;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40
ampl.param["min_total"] = 8000

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)

[WARNING] 
	presolve: constraint Commitment cannot hold:
		body >= 8000 cannot be <= 7942.86; difference = 57.1429


infeasible


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
# amplpy raises the previous solve's warning at the next statement; clear it before this run
try:
    ampl.eval("reset;")
except Exception:
    pass

In [7]:
%%writefile data.dat
set PROD := bands coils;
param:  rate  profit  market :=
bands    200     25     6000
coils    140     30     4000;
param avail := 40;
param min_total := 8000;

Writing data.dat


In [8]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [9]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
solve;
display solve_result;
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	presolve: constraint Commitment cannot hold:
		body >= 8000 cannot be <= 7942.86; difference = 57.1429


solve_result = infeasible



### Another way in amplpy

A useful answer restates the two constraints, points to the conflict between `Time` and
`Commitment`, and proposes a check before any change. AMPL's presolve message already names
`Commitment`, so include it in the prompt. You can verify that diagnosis by hand: the
largest tonnage that fits in 40 hours is 6000 tons of bands (30 hours) plus 1400 tons of coils
(10 hours), 7400 tons in all, which is less than 8000. If the model owner confirms that the
commitment should have been 7000 tons, the corrected data solves:

In [10]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [11]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.param["avail"] = 40
ampl.param["min_total"] = 7000

In [12]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print(ampl.obj["Total_Profit"].value())
print(ampl.var["Make"].to_dict())

solved
192000.0
{'bands': 6000, 'coils': 1400}
